In [1]:
from pathlib import Path
import sys

# adjust this depending on where the notebook sits
# example: notebook is in project/notebooks/experiments/
PROJECT_ROOT = Path.cwd().resolve().parents[1]   # go up 2 levels

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f"PROJECT_ROOT: \n{PROJECT_ROOT}")

import yaml
from pathlib import Path
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
import torch.nn.functional as F
import cv2
from tqdm import tqdm
import numpy as np

from datasets.bcdata import BCDataDataset, collate_heatmap_points
from datasets.transforms import PointsToLocalizationHeatmap, PointsToCountHeatmap


from visualization import overlay_heatmap, overlay_gt


from models.losses import weighted_sigmoid_mse_from_logits, softplus_mse_from_logits, l1_count_from_density_logits

from src.utils import print_info


import torch
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

PROJECT_ROOT: 
/raid/home/user6/projects/IHC/BCData_Ki67


In [2]:
with open(PROJECT_ROOT / "config.yaml", "r") as f:
    cfg = yaml.safe_load(f)

data_root = Path(cfg["h200_paths"]["data_root"])
checkpoint_dir = Path(cfg["h200_paths"]["checkpoint_dir"])
print(f"data_root: {data_root}")
print(f"checkpoint_dir: {checkpoint_dir}")

data_root: /raid/datasets/Yeldos/BCData
checkpoint_dir: checkpoints


In [3]:
loc_heatmap_generator = PointsToLocalizationHeatmap(out_hw=(160,160), in_hw=(640,640), sigma=2.0)
count_heatmap_generator = PointsToCountHeatmap(out_hw=(160,160), in_hw=(640,640), sigma=2.0)

train_dataset = BCDataDataset(root = data_root,
                        split="train",
                        target_loc_transform = loc_heatmap_generator,
                        target_count_transform = count_heatmap_generator)


test_dataset = BCDataDataset(root = data_root,
                        split="test",
                        target_loc_transform = loc_heatmap_generator,
                        target_count_transform = count_heatmap_generator)

val_dataset = BCDataDataset(root = data_root,
                        split="validation",
                        target_loc_transform = loc_heatmap_generator,
                        target_count_transform = count_heatmap_generator)

In [4]:
device = 'cpu'

In [5]:
save_dir = "./pics/train/"
for i in range(len(train_dataset)):
    img, loc_heatmap, count_heatmap, pos_pts, neg_pts = train_dataset[i]
    img = img.to(device)
    loc_heatmap = loc_heatmap.to(device)
    count_heatmap = count_heatmap.to(device)

    alpha = 0.4
    color_map = cv2.COLORMAP_RAINBOW
    overlay_gt(img, loc_heatmap, save_dir=save_dir, prefix=f"train_{i:05d}", show=False)


KeyboardInterrupt: 